# RAG Strategies

This notebook evaluates **3 chunking strategies and 6 progressive RAG configurations** on a medical QA corpus.

## Jump to Sections
- [Setup - Imports, Configuration, and Loading Chunks](#Section-0:-Setup---Imports,-Configuration,-and-Loading-Chunks)
- [Experiment 1: Chunking Strategies](#Experiment-1:-Chunking-Strategies)
- [Experiment 2: Dense vs Hybrid Retrieval](#Experiment-2:-Dense-vs-Hybrid-Retrieval)
- [Experiment 3: With vs Without Re-ranking](#Experiment-3:-With-vs-Without-Re-ranking)
- [Experiment 4: With vs Without Gradient-based Selection](#Experiment-4:-With-vs-Without-Gradient-based-Selection)
- [Experiment 5: Diverse Prompts vs K-means Vote](#Experiment-5:-Diverse-Prompts-vs-K-means-Vote)

---

## Output for Evaluation

Every main RAG run cell writes an output CSV via `output_csv` in `run_rag_loop`.
These files are saved under `data/rag_output/` (through `RAG_DIR`).

| This notebook | `medical-rag-eval.ipynb` | Description |
|---|---|---|
| `input` | `input` | Original question |
| `actual_output` | `actual_output` | RAG-generated answer |
| `retrieval_context` | `retrieval_context` | Context passed to the generator |

Examples produced by active run cells include:
- `rag2_section_400token.csv`, `rag2_fixed_500char.csv`, `rag2_section_contextual_400token.csv`
- `rag1_eval.csv`, `rag2_eval.csv`, `rag3_eval.csv`, `rag4_eval.csv`, `rag5_eval.csv`, `rag6_eval.csv`

---
# Setup - Imports, Configuration, and Loading Chunks

## 0.1 - Global Configuration

In [1]:
import os
from pathlib import Path

CWD = Path.cwd().resolve()


def _resolve_project_root(start: Path) -> Path:
    """Find the nearest parent that looks like the repo root."""
    for p in [start, *start.parents]:
        if (p / "src" / "data_loader.py").exists():
            return p
    return start


def _resolve_data_root(cwd: Path, project_root: Path) -> tuple[Path, str]:
    """Resolve data root with env override + robust auto-detection."""
    env_root = os.environ.get("RAG_DATA_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if p.exists():
            return p, "env:RAG_DATA_ROOT"
        print(f"[WARN] RAG_DATA_ROOT set but not found: {p}")

    candidates = []
    candidates.append(project_root / "data")
    for p in [cwd, *cwd.parents]:
        candidates.append(p / "data")

    # Deduplicate while preserving order
    seen = set()
    unique_candidates = []
    for c in candidates:
        key = str(c)
        if key not in seen:
            seen.add(key)
            unique_candidates.append(c)

    def score(path: Path) -> int:
        points = 0
        if (path / "bm25_index" / "section_400token.pkl").exists():
            points += 5
        if (path / "bm25_index" / "fixed_500char.pkl").exists():
            points += 5
        if (path / "vector_store" / "section_400token").exists():
            points += 5
        if (path / "vector_store" / "fixed_500char").exists():
            points += 5
        if (path / "test_final.csv").exists():
            points += 2
        if (path / "bm25_index.zip").exists():
            points += 1
        if (path / "vector_store.zip").exists():
            points += 1
        return points

    existing = [c for c in unique_candidates if c.exists()]
    if existing:
        best = max(existing, key=score)
        if score(best) > 0:
            return best.resolve(), "auto-detected"

    return (project_root / "data").resolve(), "fallback:project_root/data"


PROJECT_ROOT = _resolve_project_root(CWD)
DATA_ROOT, DATA_ROOT_SOURCE = _resolve_data_root(CWD, PROJECT_ROOT)

CSV_PATH = CWD / "test_final.csv" if (CWD / "test_final.csv").exists() \
           else DATA_ROOT / "test_final.csv"

VECTOR_DIR = DATA_ROOT / "vector_store"
BM25_DIR = DATA_ROOT / "bm25_index"
RAG_DIR = DATA_ROOT / "rag_output"
RAG_DIR.mkdir(parents=True, exist_ok=True)

N_QUESTIONS   = 300    # max 300
TOP_K_CONTEXT = 15    # final context chunks passed to generator
REFERENCE_COL = "ideal_answer"  # ground-truth column in test CSV

# RAG5 / RAG6 generation params
N_GENERATIONS = 7
TEMPERATURE   = 0.8
N_CLUSTERS    = 3

# Set automatically at the end of Section 1
BEST_CHUNK_STRATEGY = None
dense_fn_best       = None
bm25_fn_best        = None

print("CWD        :", CWD)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT  :", DATA_ROOT, f"({DATA_ROOT_SOURCE})")
print("CSV_PATH   :", CSV_PATH, "| exists?", CSV_PATH.exists())

CWD        : /workspace/notebooks
PROJECT_ROOT: /workspace
DATA_ROOT  : /workspace/data (auto-detected)
CSV_PATH   : /workspace/data/test_final.csv | exists? True


## 0.2 — Imports

In [2]:
import sys, pickle
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ── Resolve src/ ──────────────────────────────────────────────────────────────
_cwd = Path.cwd().resolve()
for _ in range(5):
    if (_cwd / "src" / "data_loader.py").exists():
        sys.path.insert(0, str(_cwd / "src"))
        print("Using src:", _cwd / "src")
        break
    _cwd = _cwd.parent
else:
    raise FileNotFoundError("Could not find src/ with data_loader.py")

from data_loader import load_and_sample_test_set
from answer_chatgpt import (
    generate_answer,
    gradient_select_chunks,
    diverse_prompted_generate,
    kmeans_majority_generate,
    format_context,
    extract_pmids
)
from bm25 import spacy_tokenize_texts          # must match tokeniser used at index build time
from chroma_manager import ChromaManager       # wraps chromadb + PubMedBERT
from hybrid import hybrid_retrieve

print("All imports loaded successfully.")

/opt/conda/lib/python3.10/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Using src: /workspace/src


/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Diverse prompted generation ready (7 personas defined).
K-Means majority vote generator ready (sentence encoder: sentence-transformers/all-MiniLM-L6-v2).

✓ Section 0 complete — all helpers loaded. Proceed to Section 1.


2026-03-21 15:01:50.011352193 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


All imports loaded successfully.


## 0.3 — Index Download & Verification

Using the output from the chunking processing,
layout after extraction:
```
data/
├── bm25_index/
│   ├── section_400token.pkl
│   ├── fixed_500char.pkl
│   └── section_contextual_400token.pkl
└── vector_store/
    ├── section_400token/pubmedbert/
    ├── fixed_500char/pubmedbert/
    └── section_contextual_400token/pubmedbert/
```


In [ ]:
# Hi prof this cell output was cleared for brevity

import zipfile

for zip_name in ["bm25_index.zip", "vector_store.zip"]:
    """Extract zip files"""
    zp = DATA_ROOT / zip_name
    if zp.exists():
        print(f"Extracting {zip_name} ...")
        with zipfile.ZipFile(zp, "r") as zf:
            zf.extractall(DATA_ROOT)
        print("  Done.")
    else:
        print(f"  {zip_name} not found — skipping (already extracted?).")

EXPECTED = [
    # BM25 indexes
    DATA_ROOT / "bm25_index" / "section_400token.pkl",
    DATA_ROOT / "bm25_index" / "fixed_500char.pkl",
    DATA_ROOT / "bm25_index" / "section_contextual_400token.pkl",  # ✅ NEW

    # Vector stores
    DATA_ROOT / "vector_store" / "section_400token",
    DATA_ROOT / "vector_store" / "fixed_500char",
    DATA_ROOT / "vector_store" / "section_contextual_400token",    # ✅ NEW
]

# Check index files
all_ok = True
for p in EXPECTED:
    ok = p.exists()
    print(f"  [{'OK' if ok else 'MISSING'}] {p.relative_to(DATA_ROOT)}")
    if not ok:
        all_ok = False


## 0.4 — Load Test Set & Build Retrievers for Both Chunk Strategies

All chunking strategies are initialised here so Section 1 can compare them without re-running setup.

For each strategy, `build_retrievers()` returns:
- `dense_fn(query, n)` — top-n results via PubMedBERT cosine similarity (`ChromaManager`)
- `bm25_fn(query, n)` — top-n results via BM25 (pre-built `.pkl` index, spaCy tokenisation)
- `chunks_df` — DataFrame of all chunks and metadata

| Strategy | Description |
|---|---|
| `section_400token`            | Section-aware chunks (~400 tokens). Preserves semantic unit boundaries.                                               |
| `fixed_500char`               | Fixed-length chunks (500 characters). May split mid-sentence.                                                         |
| `section_contextual_400token` | Section-aware chunks enriched with LLM-generated summaries. Improves semantic representation and retrieval alignment. |


In [ ]:
# load and sample test set
test_df = load_and_sample_test_set(CSV_PATH, n_total=N_QUESTIONS, random_state=42)
print("Sampled questions:", len(test_df))

ref_map = dict(zip(test_df["question"], test_df[REFERENCE_COL])) \
          if REFERENCE_COL in test_df.columns else {}


def _fetch_collection_records(collection, batch_size: int = 1000):
    """
    Fetch all ids/documents/metadatas from a Chroma collection in paged requests.
    """
    total = collection.count()
    all_ids, all_docs, all_metas = [], [], []

    for offset in range(0, total, batch_size):
        batch = collection.get(
            include=["documents", "metadatas"],
            limit=batch_size,
            offset=offset,
        )

        batch_ids = batch.get("ids") or []
        batch_docs = batch.get("documents") or []
        batch_metas = batch.get("metadatas") or [{}] * len(batch_ids)

        if len(batch_metas) < len(batch_ids):
            batch_metas = batch_metas + [{}] * (len(batch_ids) - len(batch_metas))

        all_ids.extend(batch_ids)
        all_docs.extend(batch_docs)
        all_metas.extend(batch_metas)

    return all_ids, all_docs, all_metas


def build_retrievers(chunk_strategy: str):
    """
    Initialise and return (dense_fn, bm25_fn, chunks_df) for the given
    chunk strategy.

    Dense retrieval uses ChromaManager (PubMedBERT embeddings stored in Chroma).
    BM25 retrieval loads the pre-built pickle index and tokenises queries with spaCy,
    matching the tokeniser used at index build time.

    Args:
        chunk_strategy: 'section_400token' or 'fixed_500char'

    Returns:
        dense_fn  : callable(query: str, n: int) -> list[(chunk_id, chunk_text, meta)]
        bm25_fn   : callable(query: str, n: int) -> list[(chunk_id, chunk_text, meta)]
        chunks_df : pd.DataFrame with 'chunk_text' column and metadata
    """
    # Dense retriever via ChromaManager
    chroma_db = ChromaManager(
        base_directory=VECTOR_DIR,
        chunk_strategy=chunk_strategy,
        embedding_model="pubmedbert",
        collection_name="medical_rag"
    )

    # Build chunks_df from Chroma collection (paged read to avoid SQL var limits)
    ids, docs, metas = _fetch_collection_records(chroma_db.collection, batch_size=1000)

    def _key(i):
        m = metas[i] if i < len(metas) and isinstance(metas[i], dict) else {}
        v = m.get("chunk_idx", ids[i])
        if isinstance(v, (int, float)):
            return int(v)
        if isinstance(v, str) and v.isdigit():
            return int(v)
        return i

    order = sorted(range(len(ids)), key=_key)
    chunks_df = pd.DataFrame({"chunk_text": [docs[i] for i in order]})

    meta_keys = set()
    for m in metas:
        if isinstance(m, dict):
            meta_keys.update(m.keys())

    for k in sorted(meta_keys):
        chunks_df[k] = [
            metas[i].get(k) if i < len(metas) and isinstance(metas[i], dict) else None
            for i in order
        ]

    # Prebuild ordered metadata for fast top-n retrieval returns
    ordered_metas = []
    for i in order:
        m = metas[i] if i < len(metas) and isinstance(metas[i], dict) else {}
        ordered_metas.append(m)

    def dense_fn(q: str, n: int) -> list:
        """Return top-n (chunk_id, chunk_text, meta) via PubMedBERT cosine similarity."""
        res = chroma_db.query(q, n_results=n)

        docs_res = (res.get("documents") or [[]])[0]
        metas_res = (res.get("metadatas") or [[]])[0]
        ids_res = (res.get("ids") or [[]])[0]

        out = []
        for i, d in enumerate(docs_res):
            m = metas_res[i] if i < len(metas_res) and isinstance(metas_res[i], dict) else {}
            cid = m.get("chunk_idx")
            if cid is None:
                cid = ids_res[i] if i < len(ids_res) else i
            out.append((cid, d, m))
        return out

    # BM25 retriever — pre-built pickle, spaCy tokenisation
    with open(BM25_DIR / f"{chunk_strategy}.pkl", "rb") as f:
        bm25_index = pickle.load(f)

    def bm25_fn(q: str, n: int) -> list:
        """Return top-n (chunk_id, chunk_text, meta) via BM25 lexical scoring."""
        tok = spacy_tokenize_texts([q])[0]
        scores = bm25_index.get_scores(tok)
        top = np.argsort(scores)[-n:][::-1]

        out = []
        for idx in top.tolist():
            text = chunks_df["chunk_text"].iloc[idx]
            meta = ordered_metas[idx] if idx < len(ordered_metas) else {}
            cid = meta.get("chunk_idx", idx) if isinstance(meta, dict) else idx
            out.append((cid, text, meta if isinstance(meta, dict) else {}))
        return out

    return dense_fn, bm25_fn, chunks_df


# Section-aware
print("Loading section_400token retrievers...")
dense_fn_section, bm25_fn_section, chunks_df_section = build_retrievers("section_400token")
print(f"  chunks: {len(chunks_df_section):,}")

# Fixed-length
print("Loading fixed_500char retrievers...")
dense_fn_fixed, bm25_fn_fixed, chunks_df_fixed = build_retrievers("fixed_500char")
print(f"  chunks: {len(chunks_df_fixed):,}")

# Contextual section-aware (NEW)
print("Loading section_contextual_400token retrievers...")
dense_fn_contextual, bm25_fn_contextual, chunks_df_contextual = build_retrievers("section_contextual_400token")
print(f"  chunks: {len(chunks_df_contextual):,}")

print("\nAll retrievers ready for all chunking strategies.")

Sampled questions: 147
Loading section_400token retrievers...

Loading embedding model: pritamdeka/S-PubMedBert-MS-MARCO


/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector DB ready → /workspace/data/vector_store/section_400token/pubmedbert


Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


  chunks: 109,234
Loading fixed_500char retrievers...

Loading embedding model: pritamdeka/S-PubMedBert-MS-MARCO


...

  chunks: 176,626
Loading section_contextual_400token retrievers...

Loading embedding model: pritamdeka/S-PubMedBert-MS-MARCO


/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector DB ready → /workspace/data/vector_store/section_contextual_400token/pubmedbert


Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


  chunks: 109,234

All retrievers ready for all chunking strategies.


## 0.5 — RAG Functions

In [ ]:
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(RERANKER_MODEL)

def rerank_chunks(question: str, chunks: list, top_k: int = 10) -> list:
    """
    Rerank (chunk_id, text) pairs using a cross-encoder.

    The cross-encoder scores each (query, chunk) pair jointly — query and document
    tokens attend to each other — giving more accurate relevance estimates than the
    bi-encoder used in dense retrieval. Applied only to top-N candidates (not the
    full corpus) due to latency.

    Args:
        question : User query string.
        chunks   : List of (chunk_id, chunk_text) tuples from retrieval.
        top_k    : Number of top-scoring chunks to return.

    Returns:
        List of (chunk_id, chunk_text) sorted by cross-encoder score (desc),
        truncated to top_k.
    """
    if not chunks:
        return []
    scores = reranker.predict([(question, c[1]) for c in chunks])
    return [c for c, _ in
            sorted(zip(chunks, scores), key=lambda x: x[1], reverse=True)[:top_k]]


print(f"Reranker loaded: {RERANKER_MODEL}")

Reranker loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


In [ ]:
def run_rag_loop(
    label: str,
    dense_fn,
    bm25_fn,
    chunk_strategy: str,
    use_bm25: bool      = True,
    use_reranking: bool = False,
    use_gradient: bool  = False,
    use_diverse: bool   = False,
    use_kmeans: bool    = False,
    output_csv: str     = None,
) -> pd.DataFrame:
    """
    Run retrieval + generation for one RAG configuration over the test set.

    Pipeline:
        Query
          → Dense retrieval (always, via ChromaManager/PubMedBERT)
          → BM25 retrieval  (if use_bm25; score-fused with dense via hybrid_retrieve)
          → Cross-encoder reranking (if use_reranking)
          → Gradient-based chunk pruning (if use_gradient)
          → Generation:
              K-Means majority vote   (if use_kmeans)
              Diverse persona prompts (elif use_diverse)
              Standard single prompt  (otherwise)

    Args:
        label          : Display name for this run (e.g. 'RAG3').
        dense_fn       : Dense retriever callable — (query, n) -> [(id, text)].
        bm25_fn        : BM25 retriever callable  — (query, n) -> [(id, text)].
        chunk_strategy : 'section_400token' or 'fixed_500char'.
        use_bm25       : Include BM25 in retrieval fusion.
        use_reranking  : Apply cross-encoder reranking after retrieval.
        use_gradient   : Apply gradient-based chunk pruning after reranking.
        use_diverse    : Use diverse persona prompts for generation (RAG5).
        use_kmeans     : Use K-Means majority vote for generation (RAG6).
        output_csv     : Optional path to save results CSV.

    Returns:
        pd.DataFrame with columns:
            input, generated_output, retrieved_context, rag_variant, chunk_strategy
    """
    n_pre_rerank = 20 if (use_reranking or use_gradient or use_diverse or use_kmeans) else 10
    rows = []

    for i, row in test_df.iterrows():
        q = row["question"]
        if i == 0 or (i + 1) % 10 == 0:
            print(f"  [{label}] {i+1}/{len(test_df)}")

        # Step 1 — Retrieval
        if use_bm25:
            chunks = hybrid_retrieve(
                q, dense_fn, bm25_fn,
                n_dense=70, n_bm25=50, n_final=n_pre_rerank
            )
        else:
            chunks = dense_fn(q, n_pre_rerank)

        # Step 2 — Reranking
        if use_reranking or use_gradient or use_diverse or use_kmeans:
            chunks = rerank_chunks(q, chunks, top_k=TOP_K_CONTEXT)
            
        # Step 3 — Gradient-based pruning
        if use_gradient or use_diverse or use_kmeans:
            chunks = gradient_select_chunks(q, chunks, retain_ratio=0.6)

        context = format_context(chunks, sep="\n\n---\n\n")

        pmids = extract_pmids(chunks)
        pmids_str = ";".join(pmids)

        # Step 4 — Generation
        if use_kmeans:
            answer = kmeans_majority_generate(
                q, context, n_generations=N_GENERATIONS,
                temperature=TEMPERATURE, n_clusters=N_CLUSTERS
            )
        elif use_diverse:
            answer = diverse_prompted_generate(
                q, context, n_generations=N_GENERATIONS, temperature=TEMPERATURE
            )
        else:
            #print(_model_cache)
            answer = generate_answer(
                q, context,
            )

        rows.append({
            "input":              q,
            "actual_output":      answer,      # eval notebook expects 'actual_output'
            "retrieval_context":  context,     # eval notebook expects 'retrieval_context'
            "retrieved_pmids":    pmids_str,   # ✅ NEW
            "rag_variant":        label,
            "chunk_strategy":     chunk_strategy,
        })

    df = pd.DataFrame(rows)
    if output_csv:
        df.to_csv(RAG_DIR /output_csv, index=False, encoding="utf-8")
        print(f"  Saved → {Path(output_csv).resolve()}")
    return df

---
# Experiment 1: Chunking Strategies

**Goal:** Determine the best chunking strategy to carry forward into Sections 2-5.

All three variants use the same **hybrid retrieval** pipeline (dense + BM25).
The only variable is how the corpus was chunked at index build time.

| Strategy                      | Description                                                | Expected behaviour                                                    |
| ----------------------------- | ---------------------------------------------------------- | --------------------------------------------------------------------- |
| `section_400token`            | Section-aware, ~400 tokens                                 | Preserves semantic boundaries; better for multi-sentence answers      |
| `fixed_500char`               | Fixed 500 characters                                       | Simpler; may split mid-sentence, losing context                       |
| `section_contextual_400token` | Section-aware + LLM-generated contextual summary per chunk | Enhances semantic richness; improves retrieval alignment with queries |

**Hypothesis:** `section_400token` should outperform `fixed_500char` because section boundaries preserve natural semantic units in biomedical literature.

`section_contextual_400token` is expected to perform best overall, since each chunk includes additional contextual signal that can improve both dense and sparse retrieval alignment.

In [ ]:
print("=" * 60)
print("Section 1: RAG2 — chunking strategy comparison")
print("=" * 60)

print("\n[1/3] RAG2 with section_400token ...")
df_s1_section = run_rag_loop(
    label="RAG2 (section_400token)",
    dense_fn=dense_fn_section,
    bm25_fn=bm25_fn_section,
    chunk_strategy="section_400token",
    use_bm25=True,
    output_csv="rag2_section_400token.csv"
)

print("\n[2/3] RAG2 with fixed_500char ...")
df_s1_fixed = run_rag_loop(
    label="RAG2 (fixed_500char)",
    dense_fn=dense_fn_fixed,
    bm25_fn=bm25_fn_fixed,
    chunk_strategy="fixed_500char",
    use_bm25=True,
    output_csv="rag2_fixed_500char.csv"
)

print("\n[3/3] RAG2 with section_contextual_400token ...")
df_s1_contextual = run_rag_loop(
    label="RAG2 (section_contextual_400token)",
    dense_fn=dense_fn_contextual,
    bm25_fn=bm25_fn_contextual,
    chunk_strategy="section_contextual_400token",
    use_bm25=True,
    output_csv="rag2_section_contextual_400token.csv"
)

Section 1: RAG2 — chunking strategy comparison

[1/3] RAG2 with section_400token ...
  [RAG2 (section_400token)] 1/147


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
/opt/conda/lib/python3.10/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


  [RAG2 (section_400token)] 10/147
  [RAG2 (section_400token)] 20/147
  [RAG2 (section_400token)] 30/147
  [RAG2 (section_400token)] 40/147
  [RAG2 (section_400token)] 50/147
  [RAG2 (section_400token)] 60/147
  [RAG2 (section_400token)] 70/147
  [RAG2 (section_400token)] 80/147
  [RAG2 (section_400token)] 90/147
  [RAG2 (section_400token)] 100/147
  [RAG2 (section_400token)] 110/147
  [RAG2 (section_400token)] 120/147
  [RAG2 (section_400token)] 130/147
  [RAG2 (section_400token)] 140/147
  Saved → /workspace/notebooks/rag2_section_400token.csv

[2/3] RAG2 with fixed_500char ...
  [RAG2 (fixed_500char)] 1/147


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


  [RAG2 (fixed_500char)] 10/147
  [RAG2 (fixed_500char)] 20/147
  [RAG2 (fixed_500char)] 30/147
  [RAG2 (fixed_500char)] 40/147
  [RAG2 (fixed_500char)] 50/147
  [RAG2 (fixed_500char)] 60/147
  [RAG2 (fixed_500char)] 70/147
  [RAG2 (fixed_500char)] 80/147
  [RAG2 (fixed_500char)] 90/147
  [RAG2 (fixed_500char)] 100/147
  [RAG2 (fixed_500char)] 110/147
  [RAG2 (fixed_500char)] 120/147
  [RAG2 (fixed_500char)] 130/147
  [RAG2 (fixed_500char)] 140/147
  Saved → /workspace/notebooks/rag2_fixed_500char.csv

[3/3] RAG2 with section_contextual_400token ...
  [RAG2 (section_contextual_400token)] 1/147


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


  [RAG2 (section_contextual_400token)] 10/147
  [RAG2 (section_contextual_400token)] 20/147
  [RAG2 (section_contextual_400token)] 30/147
  [RAG2 (section_contextual_400token)] 40/147
  [RAG2 (section_contextual_400token)] 50/147
  [RAG2 (section_contextual_400token)] 60/147
  [RAG2 (section_contextual_400token)] 70/147
  [RAG2 (section_contextual_400token)] 80/147
  [RAG2 (section_contextual_400token)] 90/147
  [RAG2 (section_contextual_400token)] 100/147
  [RAG2 (section_contextual_400token)] 110/147
  [RAG2 (section_contextual_400token)] 120/147
  [RAG2 (section_contextual_400token)] 130/147
  [RAG2 (section_contextual_400token)] 140/147
  Saved → /workspace/notebooks/rag2_section_contextual_400token.csv


In [ ]:
BEST_CHUNK_STRATEGY = "section_contextual_400token"
dense_fn_best = dense_fn_contextual
bm25_fn_best  = bm25_fn_contextual

After evaluating the output csvs, we have set BEST_CHUNK_STRATEGY to section_contextual_400token based on qualitative analysis of generated answers and retrieved contexts. This will be used for remaining runs."

---
# Experiment 2: Dense vs Hybrid Retrieval

**Goal:** Confirm that adding BM25 sparse retrieval improves performance over dense-only.

Both RAGs use the best chunk strategy selected in Section 1.

| RAG | Retrieval | Generation |
|-----|-----------|------------|
| RAG1 | Dense-only (PubMedBERT cosine similarity) | Standard single-prompt generation (`generate_answer`) |
| RAG2 | Hybrid (Dense + BM25 score fusion) | Standard single-prompt generation (`generate_answer`) |

**Hypothesis:** RAG2 > RAG1. BM25 captures exact biomedical term matches (e.g. gene
names, drug identifiers, rare disease names) that dense embeddings tend to miss
because they match semantically similar but lexically different passages.

In [ ]:
print("=" * 60)
print(f"Section 2: RAG1 vs RAG2  [chunk: {BEST_CHUNK_STRATEGY}]")
print("=" * 60)

print("\n[1/2] RAG1 — dense-only retrieval ...")
df_s2_rag1 = run_rag_loop(
    label="RAG1 (dense-only)",
    dense_fn=dense_fn_best, bm25_fn=bm25_fn_best,
    chunk_strategy=BEST_CHUNK_STRATEGY,
    use_bm25=False,
    output_csv="rag1_eval.csv"
)

print("\n[2/2] RAG2 — hybrid retrieval ...")
df_s2_rag2 = run_rag_loop(
    label="RAG2 (hybrid)",
    dense_fn=dense_fn_best, bm25_fn=bm25_fn_best,
    chunk_strategy=BEST_CHUNK_STRATEGY,
    use_bm25=True,
    output_csv="rag2_eval.csv"
)

Section 2: RAG1 vs RAG2  [chunk: section_contextual_400token]

[1/2] RAG1 — dense-only retrieval ...
  [RAG1 (dense-only)] 1/147
  [RAG1 (dense-only)] 10/147
  [RAG1 (dense-only)] 20/147
  [RAG1 (dense-only)] 30/147
  [RAG1 (dense-only)] 40/147
  [RAG1 (dense-only)] 50/147
  [RAG1 (dense-only)] 60/147
  [RAG1 (dense-only)] 70/147
  [RAG1 (dense-only)] 80/147
  [RAG1 (dense-only)] 90/147
  [RAG1 (dense-only)] 100/147
  [RAG1 (dense-only)] 110/147
  [RAG1 (dense-only)] 120/147
  [RAG1 (dense-only)] 130/147
  [RAG1 (dense-only)] 140/147
  Saved → /workspace/notebooks/rag1_eval.csv

[2/2] RAG2 — hybrid retrieval ...
  [RAG2 (hybrid)] 1/147
  [RAG2 (hybrid)] 10/147
  [RAG2 (hybrid)] 20/147
  [RAG2 (hybrid)] 30/147
  [RAG2 (hybrid)] 40/147
  [RAG2 (hybrid)] 50/147
  [RAG2 (hybrid)] 60/147
  [RAG2 (hybrid)] 70/147
  [RAG2 (hybrid)] 80/147
  [RAG2 (hybrid)] 90/147
  [RAG2 (hybrid)] 100/147
  [RAG2 (hybrid)] 110/147
  [RAG2 (hybrid)] 120/147
  [RAG2 (hybrid)] 130/147
  [RAG2 (hybrid)] 140/147
 

---
# Experiment 3: With vs Without Re-ranking

**Goal:** Test whether cross-encoder reranking improves the precision of context
passed to the generator.

| RAG | Retrieval | Reranking | Generation |
|-----|-----------|-----------|------------|
| RAG2 | Hybrid (Dense + BM25) | No | Standard single-prompt generation |
| RAG3 | Hybrid (Dense + BM25) | Yes (cross-encoder) | Standard single-prompt generation |

**How reranking works:** Hybrid retrieval returns 20 candidates via score fusion.
A cross-encoder (`ms-marco-MiniLM-L-6-v2`) re-scores each (query, chunk) pair
*jointly* - query and document tokens attend to each other - giving more accurate
relevance estimates than the bi-encoder. The top 15 chunks are passed to the generator.

**Hypothesis:** RAG3 > RAG2. Reranking filters out false positives from the
hybrid fusion step, giving the generator a more precisely relevant context window.

In [ ]:
print("=" * 60)
print(f"Section 3: RAG2 vs RAG3  [chunk: {BEST_CHUNK_STRATEGY}]")
print("=" * 60)

# RAG2 reused from Section 2 (df_s2_rag2)
print("\n[1/1] RAG3 — hybrid + reranking ...")
df_s3_rag3 = run_rag_loop(
    label="RAG3 (+ reranking)",
    dense_fn=dense_fn_best, bm25_fn=bm25_fn_best,
    chunk_strategy=BEST_CHUNK_STRATEGY,
    use_bm25=True, use_reranking=True,
    output_csv="rag3_eval.csv"
)

Section 3: RAG2 vs RAG3  [chunk: section_contextual_400token]

[1/1] RAG3 — hybrid + reranking ...
  [RAG3 (+ reranking)] 1/147
  [RAG3 (+ reranking)] 10/147
  [RAG3 (+ reranking)] 20/147
  [RAG3 (+ reranking)] 30/147
  [RAG3 (+ reranking)] 40/147
  [RAG3 (+ reranking)] 50/147
  [RAG3 (+ reranking)] 60/147
  [RAG3 (+ reranking)] 70/147
  [RAG3 (+ reranking)] 80/147
  [RAG3 (+ reranking)] 90/147
  [RAG3 (+ reranking)] 100/147
  [RAG3 (+ reranking)] 110/147
  [RAG3 (+ reranking)] 120/147
  [RAG3 (+ reranking)] 130/147
  [RAG3 (+ reranking)] 140/147
  Saved → /workspace/notebooks/rag3_eval.csv


---
# Experiment 4: With vs Without Gradient-based Selection

**Goal:** Test whether pruning low-salience context chunks further improves
answer accuracy after reranking.

| RAG | Retrieval | Reranking | Context Pruning | Generation |
|-----|-----------|-----------|-----------------|------------|
| RAG3 | Hybrid | Yes (cross-encoder) | No | Standard single-prompt generation |
| RAG4 | Hybrid | Yes (cross-encoder) | Yes (gradient saliency) | Standard single-prompt generation |

**How gradient-based selection works:** After reranking, 15 chunks are concatenated
as context. Gradient-based selection runs a forward+backward pass through the generator
and computes per-token saliency:

$$s_i = \left\|\frac{\partial \mathcal{L}}{\partial e_i}\right\|_2$$

Chunks are ranked by mean token saliency and the bottom 40% are dropped
(`retain_ratio = 0.6`), leaving about 9 higher-signal chunks for generation.

**Trade-off:** Requires one additional forward+backward pass per question
(approximately 2x inference time vs. RAG3).

**Hypothesis:** RAG4 > RAG3 (modest gain). Removing tangential context reduces
the chance of the generator being misled by irrelevant passages.

In [ ]:
print("=" * 60)
print(f"Section 4: RAG3 vs RAG4  [chunk: {BEST_CHUNK_STRATEGY}]")
print("=" * 60)

# RAG3 reused from Section 3 (df_s3_rag3)
print("\n[1/1] RAG4 — hybrid + reranking + gradient selection ...")
df_s4_rag4 = run_rag_loop(
    label="RAG4 (+ gradient selection)",
    dense_fn=dense_fn_best, bm25_fn=bm25_fn_best,
    chunk_strategy=BEST_CHUNK_STRATEGY,
    use_bm25=True, use_reranking=True, use_gradient=True,
    output_csv="rag4_eval.csv"
)

Section 4: RAG3 vs RAG4  [chunk: section_contextual_400token]

[1/1] RAG4 — hybrid + reranking + gradient selection ...
  [RAG4 (+ gradient selection)] 1/147
🔥 Loading model: Qwen/Qwen2.5-1.5B-Instruct
  [RAG4 (+ gradient selection)] 10/147
  [RAG4 (+ gradient selection)] 20/147
  [RAG4 (+ gradient selection)] 30/147
  [RAG4 (+ gradient selection)] 40/147
  [RAG4 (+ gradient selection)] 50/147
  [RAG4 (+ gradient selection)] 60/147
  [RAG4 (+ gradient selection)] 70/147
  [RAG4 (+ gradient selection)] 80/147
  [RAG4 (+ gradient selection)] 90/147
  [RAG4 (+ gradient selection)] 100/147
  [RAG4 (+ gradient selection)] 110/147
  [RAG4 (+ gradient selection)] 120/147
  [RAG4 (+ gradient selection)] 130/147
  [RAG4 (+ gradient selection)] 140/147
  Saved → /workspace/notebooks/rag4_eval.csv


---
# Experiment 5: Diverse Prompts vs K-means Vote

**Goal:** Compare two advanced generation strategies against the RAG4 greedy baseline.
All three share the same retrieval pipeline (hybrid + reranking + gradient selection).
The only difference is the generation strategy.

| RAG | Retrieval | Generation strategy |
|-----|-----------|--------------------|
| RAG4 | Hybrid + Reranking + Gradient | Single prompt via `generate_answer` |
| RAG5 | Hybrid + Reranking + Gradient | 7 expert persona prompts, temperature sampling, token-overlap vote |
| RAG6 | Hybrid + Reranking + Gradient | N samples, sentence embeddings, K-Means clustering, centroid vote |



### RAG5 - Diverse Prompted Generation
Uses 7 expert persona prompts (*medical assistant, geneticist, clinical researcher,
rare-disease specialist, EBM reviewer, paediatric specialist, epidemiologist*)
sampled with the shared temperature setting (`TEMPERATURE`). The final answer is
the candidate with the highest average **token-overlap** with all others
(most semantically central).

**Hypothesis:** Diverse prompts surface rare domain knowledge that a single generic
prompt underweights, improving recall on specialist medical questions.



### RAG6 - K-Means Majority Vote
Generates N = 7 responses with the shared temperature setting (`TEMPERATURE`),
encodes them with `all-MiniLM-L6-v2`, clusters with K-Means (k = 3), and selects
the response closest to the centroid of the largest cluster.

Unlike RAG5's token-overlap vote (lexical), K-Means operates in embedding space
and is robust to paraphrasing - semantically equivalent answers cluster together
regardless of surface wording.

**Hypothesis:** RAG6 > RAG5 > RAG4 overall. Embedding-space clustering is a more
principled form of majority voting and better handles lexical variation.

In [ ]:
print("=" * 60)
print(f"Section 5: RAG4 vs RAG5 vs RAG6  [chunk: {BEST_CHUNK_STRATEGY}]")
print("=" * 60)

# RAG4 reused from Section 4 (df_s4_rag4)

print("\n[1/2] RAG5 — diverse prompted generation ...")
df_s5_rag5 = run_rag_loop(
    label="RAG5 (diverse prompts)",
    dense_fn=dense_fn_best, bm25_fn=bm25_fn_best,
    chunk_strategy=BEST_CHUNK_STRATEGY,
    use_bm25=True, use_reranking=True, use_gradient=True, use_diverse=True,
    output_csv="rag5_eval.csv"
)

print("\n[2/2] RAG6 — K-Means majority vote ...")
df_s5_rag6 = run_rag_loop(
    label="RAG6 (k-means vote)",
    dense_fn=dense_fn_best, bm25_fn=bm25_fn_best,
    chunk_strategy=BEST_CHUNK_STRATEGY,
    use_bm25=True, use_reranking=True, use_gradient=True, use_kmeans=True,
    output_csv="rag6_eval.csv"
)

Section 5: RAG4 vs RAG5 vs RAG6  [chunk: section_contextual_400token]

[1/2] RAG5 — diverse prompted generation ...
  [RAG5 (diverse prompts)] 1/147
  [RAG5 (diverse prompts)] 10/147
  [RAG5 (diverse prompts)] 20/147
  [RAG5 (diverse prompts)] 30/147
  [RAG5 (diverse prompts)] 40/147
  [RAG5 (diverse prompts)] 50/147
  [RAG5 (diverse prompts)] 60/147
  [RAG5 (diverse prompts)] 70/147
  [RAG5 (diverse prompts)] 80/147
  [RAG5 (diverse prompts)] 90/147
  [RAG5 (diverse prompts)] 100/147
  [RAG5 (diverse prompts)] 110/147
  [RAG5 (diverse prompts)] 120/147
  [RAG5 (diverse prompts)] 130/147
  [RAG5 (diverse prompts)] 140/147
  Saved → /workspace/notebooks/rag5_eval.csv

[2/2] RAG6 — K-Means majority vote ...
  [RAG6 (k-means vote)] 1/147


/opt/conda/lib/python3.10/site-packages/sklearn/base.py:1474: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/opt/conda/lib/python3.10/site-packages/sklearn/base.py:1474: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/opt/conda/lib/python3.10/site-packages/sklearn/base.py:1474: ConvergenceWarning: Number of distinct clusters (2) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/opt/conda/lib/python3.10/site-packages/sklearn/base.py:1474: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


  [RAG6 (k-means vote)] 10/147


/opt/conda/lib/python3.10/site-packages/sklearn/base.py:1474: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/opt/conda/lib/python3.10/site-packages/sklearn/base.py:1474: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/opt/conda/lib/python3.10/site-packages/sklearn/base.py:1474: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/opt/conda/lib/python3.10/site-packages/sklearn/base.py:1474: ConvergenceWarning: Number of distinct clusters (2) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/opt/conda/lib/python3.10/site-packages/sklearn/base.py:

  [RAG6 (k-means vote)] 20/147


...

/opt/conda/lib/python3.10/site-packages/sklearn/base.py:1474: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/opt/conda/lib/python3.10/site-packages/sklearn/base.py:1474: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/opt/conda/lib/python3.10/site-packages/sklearn/base.py:1474: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/opt/conda/lib/python3.10/site-packages/sklearn/base.py:1474: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/opt/conda/lib/python3.10/site-packages/sklearn/base.py:

  [RAG6 (k-means vote)] 140/147


/opt/conda/lib/python3.10/site-packages/sklearn/base.py:1474: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/opt/conda/lib/python3.10/site-packages/sklearn/base.py:1474: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


  Saved → /workspace/notebooks/rag6_eval.csv


/opt/conda/lib/python3.10/site-packages/sklearn/base.py:1474: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
